In [1]:
#IMPORTS CELL

import re
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter
from scipy import stats                      # linear regression (Lotka) and Kendall's tau
from sqlalchemy import func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons: standardised name, sector, country
    TLS207_PERS_APPLN,          # person <-> application links (applicant / inventor)
    TLS209_APPLN_IPC,           # IPC classification
    TLS211_PAT_PUBLN,           # publications: grants and their dates
    TLS224_APPLN_CPC,           # CPC classification
)

In [2]:
#DB SETUP

# ---------------------------------------------------------------- environment
ENV = "TEST"                         # develop on TEST, switch to "PROD" for the final numbers
OUTPUT_DIR = Path("outputs/step2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- applicant names (Section 2)
NAME_METHOD = "dictionary"           # "psn" (PATSTAT names as they are), "dictionary" or "cleanco"
GOLD_STANDARD_FILE = "PATSTAT_500_label_datav4.csv"   # hand-labelled names (Section 2.4), in the same folder as this notebook

# ---------------------------------------------------------------- replication (Section 4): the paper's design
REPL_OFFICE = "US"                                      # USPTO patents...
REPL_GRANTED = (date(1976, 1, 1), date(2002, 12, 31))   # ...granted between these two dates
REPL_KEYWORDS = ("fuel cell",)                          # searched in titles and abstracts
REPL_CLASSES = ("H01M8",)                               # IPC/CPC H01M 8: fuel cells
REPL_SELECTION = "keywords or classes"                  # "keywords", "classes" or "keywords or classes"
PAPER_COUNTRIES = ("AU", "CA", "FR", "DE", "IL", "IT", "JP", "NL", "SE", "CH", "GB", "US")

# ---------------------------------------------------------------- your sector (Section 5): replace with your Step 1
SECTOR_NAME = "Fuel cells"
SECTOR_CPC_PREFIXES = (
    "H01M8",                 # fuel cells; manufacture thereof
    "Y02E60/50",             # fuel cells (climate change mitigation tagging)
)
SECTOR_KEYWORDS = ()         # optional, e.g. ("fuel cell",)
YEAR_MIN, YEAR_MAX = 2000, 2025
LAST_COMPLETE_YEAR = YEAR_MAX - 3    # the last 3 priority years are still incomplete in PATSTAT
IPR_TYPE = "PI"                      # patents of invention only
SPLIT_YEAR = 2015                    # incumbents file before SPLIT_YEAR, entrants from SPLIT_YEAR on
WINDOW = 3                           # years per window when following concentration over time
TOP_N = 15                           # rows in the tables of top applicants

COUNTRY_NAMES = {"AU": "Australia", "CA": "Canada", "FR": "France", "DE": "Germany", "IL": "Israel", "IT": "Italy",
                 "JP": "Japan", "NL": "Netherlands", "SE": "Sweden", "CH": "Switzerland", "GB": "UK", "US": "USA"}

In [3]:
patstat = PatstatClient(env=ENV)
db = patstat.orm()

In [4]:
#--1: simple manual cleanup

In [5]:
# Legal-form words to remove (extend the set for the countries of your sector)
LEGAL_FORMS = {
    "INC",
    "INCORPORATED",
    "CORP",
    "CORPORATION",
    "CO",
    "COMPANY",
    "LTD",
    "LIMITED",
    "LLC",
    "LP",
    "PLC",
    "GMBH",
    "AG",
    "KG",
    "MBH",
    "SE",
    "SA",
    "SAS",
    "SPA",
    "SRL",
    "BV",
    "NV",
    "AB",
    "AS",
    "ASA",
    "OY",
    "OYJ",
    "KK",
    "KABUSHIKI",
    "KAISHA",
    "LTDA",
    "PTY",
    "THE",
    "AKTIENGESELLSCHAFT",
    "GESELLSCHAFT",
}


def clean_with_dictionary(name):
    """Upper-case, keep letters and digits only, and drop legal-form words at the start and at the end."""
    words = re.sub(r"[^A-Z0-9 ]", " ", str(name).upper()).split()
    while words and words[-1] in LEGAL_FORMS:
        words.pop()
    while words and words[0] in LEGAL_FORMS:
        words.pop(0)
    return " ".join(words)

In [6]:
#--2: Cleanco + manual cleanup

In [7]:
%pip install -q cleanco

Note: you may need to restart the kernel to use updated packages.


In [8]:
from cleanco import basename


def clean_with_cleanco(name):
    "Remove legal forms with cleanco, then apply the same upper-case cleaning as method A."
    return clean_with_dictionary(basename(str(name)))



In [9]:
# ---------------------------------------------------------------- Entity Linking Group Validation

# 1. Define target group column name in your CSV (e.g., 'doc_std_name' or 'group_id')
COL_RAW = "comp"
COL_GROUP = "Group"

# 2. Load ground truth and build normalized lookups
df_gold = pd.read_csv(GOLD_STANDARD_FILE)
df_gold["clean_key"] = df_gold[COL_RAW].apply(clean_with_dictionary)

# Mapping dictionary: clean name -> most common assigned group
name_to_group = (
    df_gold.groupby("clean_key")[COL_GROUP]
    .agg(lambda x: x.mode()[0] if not x.empty else None)
    .to_dict()
)

# 3. Predict group assignment and evaluate
df_gold["predicted_group"] = df_gold["clean_key"].map(name_to_group)
df_gold["is_correct"] = df_gold["predicted_group"] == df_gold[COL_GROUP]

accuracy = df_gold["is_correct"].mean()
print(
    f"Group Assignment Accuracy ({NAME_METHOD}): {accuracy:.2%} ({df_gold['is_correct'].sum()}/{len(df_gold)})"
)

# 4. Display misclassifications
df_errors = df_gold[~df_gold["is_correct"]][
    [COL_RAW, "clean_key", COL_GROUP, "predicted_group"]
]
print(f"\nTop Misclassified Cases ({len(df_errors)} total):")
display(df_errors.head(10))

Group Assignment Accuracy (dictionary): 99.97% (3191/3192)

Top Misclassified Cases (1 total):


,comp,clean_key,Group,predicted_group
2610,SUMITOMO CHEMICAL COMPANY,SUMITOMO CHEMICAL,375,373


In [10]:
#FOLLOWING CODE WILL BE SUBSTITUTED BY TERRORIZER WHEN AVAILABLE
#AS TERRORIZER IS THE BEST APPROACH IN LITERATURE TO SOLVE THE ENTITY LINKING PROBLEM
#FOR PATENT APPLICANTS

In [11]:
%pip install networkx sentence_transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 33.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 57.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 125.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 169.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 164.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.8/807.8 kB 88.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 52.2 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 50.3 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 83.2 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 78.4 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 58.9 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 56.6 MB/s  0:00:00
   ━━━━━━━

In [12]:
import networkx as nx
import networkx.algorithms.community as nx_comm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. MATCHING ENGINE: Embeddings & Similarity
print("Loading SentenceTransformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract unique cleaned names
unique_names = df_gold["clean_key"].dropna().unique()

print(f"Computing embeddings for {len(unique_names)} unique aliases...")
embeddings = model.encode(unique_names, show_progress_bar=True)
sim_matrix = cosine_similarity(embeddings)

# 2. FILTERING: Network Graph Construction
print("Building similarity graph...")
G = nx.Graph()
G.add_nodes_from(unique_names)

SIM_THRESHOLD = 0.75 

# Vectorized edge creation (faster and simpler than nested loops)
i_idx, j_idx = np.where(np.triu(sim_matrix, k=1) >= SIM_THRESHOLD)
edges = [(unique_names[i], unique_names[j], sim_matrix[i, j]) for i, j in zip(i_idx, j_idx)]
G.add_weighted_edges_from(edges)

# 3. CLUSTERING: Louvain Community Detection
print("Extracting communities using Louvain algorithm...")
communities = nx_comm.louvain_communities(G, weight='weight', resolution=1.0)

# Map each alias to its assigned cluster ID
alias_to_cluster = {
    alias: cluster_id 
    for cluster_id, comm in enumerate(communities) 
    for alias in comm
}

# 4. ASSIGNMENT & RESULTS
df_gold["predicted_terrorizer_group"] = df_gold["clean_key"].map(alias_to_cluster)

print(f"\nResult: {len(unique_names)} original aliases reduced to {len(communities)} corporate clusters.")

# Display top 5 extracted clusters
print("\nSample of extracted clusters:")
for i, comm in enumerate(list(communities)[:5]):
    print(f"Cluster {i}: {list(comm)}")

Loading SentenceTransformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing embeddings for 3010 unique aliases...


Batches:   0%|          | 0/95 [00:00<?, ?it/s]

Building similarity graph...
Extracting communities using Louvain algorithm...

Result: 3010 original aliases reduced to 1279 corporate clusters.

Sample of extracted clusters:
Cluster 0: ['NOKIAN TYRES']
Cluster 1: ['NOKIA MOBILE DEVELOPMENTS LIMITED C O CLYDE']
Cluster 2: ['NOKIAN CAPACITORS']
Cluster 3: ['NOKIA ALUMIINI']
Cluster 4: ['NOKIA MECHATRONICS']


In [13]:
# 5. EVALUATION & COMPARISON

# Map each (homemade) Terrorizer cluster ID to the most frequent Ground Truth group inside it (done just to make labels comaprale with GTs of gold standard)
cluster_to_gt = (
    df_gold.groupby("predicted_terrorizer_group")[COL_GROUP]
    .agg(lambda x: x.mode()[0] if not x.empty else None)
    .to_dict()
)

# Apply the mapped labels back to the dataframe
df_gold["terrorizer_mapped_group"] = df_gold["predicted_terrorizer_group"].map(cluster_to_gt)

# Calculate Terrorizer Accuracy
df_gold["is_terrorizer_correct"] = df_gold["terrorizer_mapped_group"] == df_gold[COL_GROUP]
terrorizer_accuracy = df_gold["is_terrorizer_correct"].mean()

# Calculate Previous Baseline Accuracy (assuming 'predicted_group' is still in df_gold)
baseline_accuracy = df_gold["is_correct"].mean()

# ------------------------------------------------------------------------------
# Display Comparison
# ------------------------------------------------------------------------------
print("=== PERFORMANCE COMPARISON ===")
print(f"1. Dictionary Baseline Accuracy: {baseline_accuracy:.2%} ({df_gold['is_correct'].sum()}/{len(df_gold)})")
print(f"2. Terrorizer Graph Accuracy:    {terrorizer_accuracy:.2%} ({df_gold['is_terrorizer_correct'].sum()}/{len(df_gold)})\n")

# Show where Terrorizer succeeded but Baseline failed
improvements = df_gold[(df_gold["is_terrorizer_correct"]) & (~df_gold["is_correct"])]
print(f"Cases fixed by Terrorizer ({len(improvements)}):")
display(improvements[[COL_RAW, "clean_key", COL_GROUP, "terrorizer_mapped_group"]].head(5))

# Show where Terrorizer failed (Useful for tuning SIM_THRESHOLD)
terr_errors = df_gold[~df_gold["is_terrorizer_correct"]]
print(f"\nRemaining Terrorizer Errors ({len(terr_errors)}):")
display(terr_errors[[COL_RAW, "clean_key", COL_GROUP, "terrorizer_mapped_group"]].head(5))

=== PERFORMANCE COMPARISON ===
1. Dictionary Baseline Accuracy: 99.97% (3191/3192)
2. Terrorizer Graph Accuracy:    94.86% (3028/3192)

Cases fixed by Terrorizer (0):


,comp,clean_key,Group,terrorizer_mapped_group



Remaining Terrorizer Errors (164):


,comp,clean_key,Group,terrorizer_mapped_group
3,NOKIA SIEMENS NETWORKS SCHWEIZ,NOKIA SIEMENS NETWORKS SCHWEIZ,2,1
6,NOKIA SIEMENS NETWORKS S.P.A.,NOKIA SIEMENS NETWORKS S P A,2,1
7,NOKIA SIEMENS NETWORKS,NOKIA SIEMENS NETWORKS,2,1
15,NOKIA SIEMENS NETWORKS ASSET MANAGEMENT,NOKIA SIEMENS NETWORKS ASSET MANAGEMENT,2,1
34,NOKIAN RENKAAT OYJ.,NOKIAN RENKAAT,3,1
